# Interview Prep — Medium Questions


## P1: MinStack — O(1) min()

In [ ]:
class MinStack:
    def __init__(self):
        self._stack = []
        self._min_stack = []

    def push(self, val: int) -> None:
        self._stack.append(val)
        min_val = min(val, self._min_stack[-1]) if self._min_stack else val
        self._min_stack.append(min_val)

    def pop(self) -> int:
        self._min_stack.pop()
        return self._stack.pop()

    def top(self) -> int:
        return self._stack[-1]

    def get_min(self) -> int:
        return self._min_stack[-1]

s = MinStack()
for v in [5, 3, 7, 2, 8, 1]:
    s.push(v)
    print(f'push({v}) -> min={s.get_min()}')

print()
while s._stack:
    print(f'pop({s.top()}) -> min={s.get_min() if len(s._min_stack)>1 else "empty"}')
    s.pop()

## P2: LRU Cache

In [ ]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            evicted = self.cache.popitem(last=False)
            print(f'  Evicted: {evicted}')

cache = LRUCache(3)
cache.put(1, 'a'); cache.put(2, 'b'); cache.put(3, 'c')
print(f'get(1): {cache.get(1)}')  # a, moves 1 to end
cache.put(4, 'd')                  # evicts 2 (LRU)
print(f'get(2): {cache.get(2)}')  # -1 (evicted)
print(f'get(3): {cache.get(3)}')  # c
print(f'get(4): {cache.get(4)}')  # d

## P3: Binary Search

In [ ]:
def binary_search(arr: list[int], target: int) -> int:
    lo, hi = 0, len(arr) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

arr = list(range(0, 20, 2))  # [0,2,4,...,18]
print(f'Array: {arr}')
for target in [0, 10, 18, 7, 20]:
    idx = binary_search(arr, target)
    print(f'  search({target}) -> index {idx}')

## P4: Group Anagrams

In [ ]:
from collections import defaultdict

def group_anagrams(strs: list[str]) -> list[list[str]]:
    groups = defaultdict(list)
    for s in strs:
        key = tuple(sorted(s))
        groups[key].append(s)
    return list(groups.values())

words = ['eat','tea','tan','ate','nat','bat']
result = group_anagrams(words)
for group in sorted(result, key=lambda g: g[0]):
    print(sorted(group))

## P5: Longest Substring Without Repeating Characters

In [ ]:
def length_of_longest_substring(s: str) -> int:
    """Sliding window — O(n) time, O(min(n,m)) space."""
    char_index = {}
    max_len = start = 0
    for i, ch in enumerate(s):
        if ch in char_index and char_index[ch] >= start:
            start = char_index[ch] + 1
        char_index[ch] = i
        max_len = max(max_len, i - start + 1)
    return max_len

test_cases = [
    ('abcabcbb', 3),
    ('bbbbb', 1),
    ('pwwkew', 3),
    ('', 0),
    ('abcdef', 6),
]
for s, expected in test_cases:
    result = length_of_longest_substring(s)
    status = 'PASS' if result == expected else 'FAIL'
    print(f'{status}  "{s}" -> {result} (expected {expected})')

## P6: Number of Islands (DFS)

In [ ]:
def num_islands(grid: list[list[str]]) -> int:
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    count = 0

    def dfs(r, c):
        if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] != '1':
            return
        grid[r][c] = '#'  # mark visited
        for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
            dfs(r+dr, c+dc)

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                dfs(r, c)
                count += 1
    return count

grids = [
    (['11110','11010','11000','00000'], 1),
    (['11000','11000','00100','00011'], 3),
]
for grid_strs, expected in grids:
    grid = [list(row) for row in grid_strs]
    result = num_islands(grid)
    print(f'Islands: {result} (expected {expected}) -> {"PASS" if result==expected else "FAIL"}')

## P7: Merge Intervals

In [ ]:
def merge_intervals(intervals: list[list[int]]) -> list[list[int]]:
    if not intervals:
        return []
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

assert merge_intervals([[1,3],[2,6],[8,10],[15,18]]) == [[1,6],[8,10],[15,18]]
assert merge_intervals([[1,4],[4,5]]) == [[1,5]]
print('Merge intervals tests passed')

## P8: Implement a Decorator with Arguments

In [ ]:
import time
from functools import wraps

def retry(max_attempts=3, delay=0.1, exceptions=(Exception,)):
    """Retry decorator with exponential backoff."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exc = e
                    if attempt < max_attempts:
                        wait = delay * (2 ** (attempt - 1))
                        print(f'Attempt {attempt} failed: {e}. Retrying in {wait:.2f}s...')
                        time.sleep(wait)
            raise last_exc
        return wrapper
    return decorator

call_count = 0

@retry(max_attempts=3, delay=0.01)
def flaky():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ValueError(f'Failed on attempt {call_count}')
    return 'Success!'

result = flaky()
print(f'Result: {result} (took {call_count} attempts)')

## P9: Thread-Safe Counter

In [ ]:
import threading

class ThreadSafeCounter:
    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()

    def increment(self, n=1):
        with self._lock:
            self._value += n

    @property
    def value(self):
        with self._lock:
            return self._value

counter = ThreadSafeCounter()
threads = [threading.Thread(target=counter.increment) for _ in range(1000)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Counter value: {counter.value} (expected 1000)')
assert counter.value == 1000

## P10: Implement Context Manager

In [ ]:
import time

class Timer:
    """Context manager that measures execution time."""

    def __init__(self, label=''):
        self.label = label
        self.elapsed = 0.0

    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self._start
        label = f'{self.label}: ' if self.label else ''
        print(f'{label}{self.elapsed*1000:.2f}ms')
        return False  # don't suppress exceptions

with Timer('list comprehension') as t:
    result = [x**2 for x in range(100_000)]
print(f'Elapsed: {t.elapsed:.4f}s')

with Timer('generator sum'):
    result = sum(x**2 for x in range(100_000))